In [13]:
import sys 
sys.path.append('../')
import argparse
import shutil
import pytorch_lightning as pl
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from models.classical_ml import get_ml_models, get_param_grids
from dataloaders.ml_dataloaders import get_dataloaders_ml, get_classical_test_loader_center2
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
from pathlib import Path
import os
import pandas as pd
import shutil
import argparse
import shap
import numpy as np
pl.seed_everything(42)

def train_ml_model(argparse, fold_index: int):
    X_train, y_train, X_test, y_test = get_dataloaders_ml(argparse, fold_index)
    center2_X_test, center2_y_test = get_classical_test_loader_center2(argparse)
    models = get_ml_models()
    
    results = []
    fold_results = {}

    center2_results = []
    center2_fold_results = {}
    
    name = "AdaBoost"
    model = models[name]
    # grid search for hyperparameter tuning
    param_grids = get_param_grids()
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    print(f"Tuning and training: {name}")

    param_grid = param_grids.get(name, None)

    if param_grid is not None:
        search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            refit=True
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
        best_inner_score = search.best_score_
    else:
        best_model = model
        best_model.fit(X_train, y_train)
        best_params = {}
        best_inner_score = None

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    metrics = compute_classification_metrics(name, y_test, y_pred, y_prob)
    results.append(metrics)

    # Center2 test
    center2_y_pred = best_model.predict(center2_X_test)
    center2_y_prob = best_model.predict_proba(center2_X_test)[:, 1]
    center2_metrics = compute_classification_metrics(name, center2_y_test, center2_y_pred, center2_y_prob)
    center2_results.append(center2_metrics)

    df_results = pd.DataFrame(results)
    center2_df_results = pd.DataFrame(center2_results)
    print(f"Fold {fold_index} results center 1:")
    display(df_results)
    print(f"Fold {fold_index} results center 2:")
    display(center2_df_results)

    # ---- SHAP on Center 2 ----
    feature_names = getattr(center2_X_test, "columns", [f"f{i}" for i in range(center2_X_test.shape[1])])

    # use a small background set for speed
    if isinstance(X_train, pd.DataFrame):
        background = X_train.sample(min(100, len(X_train)), random_state=42)
    else:
        idx = np.random.RandomState(42).choice(len(X_train), size=min(100, len(X_train)), replace=False)
        background = X_train[idx]

    explainer = shap.Explainer(
    best_model.predict_proba,
    background,
    max_evals=2 * center2_X_test.shape[1] + 1
)
    shap_values = explainer(center2_X_test)

    # binary classification -> take SHAP for positive class
    values = shap_values.values[..., 1]   # shape: [n_samples, n_features]
    mean_abs_shap = np.abs(values).mean(axis=0)

    shap_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    print(f"Top SHAP features for fold {fold_index} on Center 2:")
    display(shap_df.head(20))

    shap_df["fold"] = fold_index
    shap_save_dir = Path("center2_shap")
    shap_save_dir.mkdir(parents=True, exist_ok=True)
    shap_df.to_csv(shap_save_dir / f"shap_fold_{fold_index}.csv", index=False)



Seed set to 42


In [14]:
class Args:
    def __init__(self):
        self.data_root = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset"
        self.use_coords = True
        self.use_demographic = True

args = Args()
train_ml_model(args, fold_index=0)

/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/cleaned_code/notebooks/../dataloaders/ml_dataloaders.py:39: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew_vals = scipy.stats.skew(


Tuning and training: AdaBoost
Fold 0 results center 1:


,model,accuracy,precision,recall,specificity,f1_score,roc_auc
0,AdaBoost,0.774194,0.727273,0.941176,0.571429,0.820513,0.836134


Fold 0 results center 2:


,model,accuracy,precision,recall,specificity,f1_score,roc_auc
0,AdaBoost,0.7375,0.647059,0.916667,0.590909,0.758621,0.840909


PermutationExplainer explainer: 81it [20:03, 15.04s/it]                        

Top SHAP features for fold 0 on Center 2:


,feature,mean_abs_shap
324,f324,0.048883
556,f556,0.041430
108,f108,0.032175
361,f361,0.028908
157,f157,0.024653
444,f444,0.023681
332,f332,0.019674
243,f243,0.013059
29,f29,0.008913
288,f288,0.008073


In [15]:
import ast
import json

AGG_NAMES = ["mean", "std", "median", "max", "min", "skew"]

def read_json(file_path):   
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def get_base_feature_names(example_radiomics_dict, use_coords=False, use_demographic=False):
    # preserve the same order as in your code: list(radiomics_features.values())
    first_node_key = list(example_radiomics_dict.keys())[0]
    first_node_features = example_radiomics_dict[first_node_key]

    base_names = list(first_node_features.keys())

    if use_demographic:
        base_names += ["Gender", "Age"]

    if use_coords:
        base_names += ["CoordX", "CoordY", "CoordZ"]

    return base_names


def get_aggregated_feature_names(example_radiomics_dict, use_coords=False, use_demographic=False):
    base_names = get_base_feature_names(
        example_radiomics_dict,
        use_coords=use_coords,
        use_demographic=use_demographic
    )

    aggregated_names = []
    for agg in AGG_NAMES:
        aggregated_names += [f"{agg}__{name}" for name in base_names]

    return aggregated_names

radiomics_path = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset/Masih-SUV/ABAZARI SHIMA 94042007/radiomics/radiomics_each_lesion.json"
radiomics_features_dict = read_json(radiomics_path)

feature_names = get_aggregated_feature_names(
    radiomics_features_dict,
    use_coords=True,
    use_demographic=True
)

print(len(feature_names))
print(feature_names[:20])

672
['mean__original_shape_Elongation', 'mean__original_shape_Flatness', 'mean__original_shape_LeastAxisLength', 'mean__original_shape_MajorAxisLength', 'mean__original_shape_Maximum2DDiameterColumn', 'mean__original_shape_Maximum2DDiameterRow', 'mean__original_shape_Maximum2DDiameterSlice', 'mean__original_shape_Maximum3DDiameter', 'mean__original_shape_MeshVolume', 'mean__original_shape_MinorAxisLength', 'mean__original_shape_Sphericity', 'mean__original_shape_SurfaceArea', 'mean__original_shape_SurfaceVolumeRatio', 'mean__original_shape_VoxelVolume', 'mean__original_firstorder_10Percentile', 'mean__original_firstorder_90Percentile', 'mean__original_firstorder_Energy', 'mean__original_firstorder_Entropy', 'mean__original_firstorder_InterquartileRange', 'mean__original_firstorder_Kurtosis']


In [22]:
# read shap values
shap_dir = Path("center2_shap")
df = pd.read_csv(shap_dir/"shap_fold_0.csv")
# assign feature names to shap values based on featur column in df
def map_shap_feature_names(shap_df, feature_names):
    shap_df = shap_df.copy()

    # extract index from fXXX
    shap_df["feature_idx"] = shap_df["feature"].str.replace("f", "").astype(int)

    # map to real names
    shap_df["feature_name"] = shap_df["feature_idx"].apply(lambda i: feature_names[i])

    return shap_df
shap_df = map_shap_feature_names(df, feature_names)
display(shap_df[:20])

,feature,mean_abs_shap,fold,feature_idx,feature_name
0,f324,0.048883,0,324,median__original_gldm_SmallDependenceHighGrayL...
1,f556,0.041430,0,556,min__Age
2,f108,0.032175,0,108,mean__Age
3,f361,0.028908,0,361,max__original_firstorder_Range
4,f157,0.024653,0,157,std__original_glcm_Imc1
5,f444,0.023681,0,444,max__Age
6,f332,0.019674,0,332,median__Age
7,f243,0.013059,0,243,median__original_firstorder_Kurtosis
8,f29,0.008913,0,29,mean__original_firstorder_TotalEnergy
9,f288,0.008073,0,288,median__original_glrlm_RunEntropy
